In [1]:
import importlib
import weights_cuda
import spike_engine_cuda
importlib.reload(weights_cuda)
importlib.reload(spike_engine_cuda)
from spike_engine_cuda import SpikeEngineCUDA
from topologies import square_torus
import cupy as cp


In [78]:
# N = 50, lifetime = 100000 # can it finish in a minute?
N = 256
lifetime = 50000
engine = SpikeEngineCUDA(
    square_torus(N),
    (N, N),
    use_k2tree=True,
    verify_k2tree=False,
    verify_progress_every=2000,
    decay_rate=0.949,
    rank=64,
    #weight_initializer=lambda size: cp.zeros(size)
)


Constructing weight matrix...
Weights constructed.


In [79]:
# Estimate bifurcation threshold and set constant weights near it
w_accum, w_instant = engine.estimate_bifurcation_weight(input_period=1)
target, _, _ = engine.set_constant_weights_near_bifurcation(input_period=1, scale=1.052, freeze_learning=True)
print(f"w_accum={w_accum:.6f} w_instant={w_instant:.6f} target={target:.6f}")


w_accum=0.854100 w_instant=0.900000 target=0.898513


In [80]:
input_neuron = (N * N) // 2 + N//2
engine.set_input_neurons([input_neuron])
inputspikes = cp.ones((lifetime, 1))

engine.start_static_record(
    inputspikes,
    lifetime,
    "cuda_test_9.spire.gz",
    record_membrane=True,
    full_decay=True,
    compression_level=4,
    compression_async=True,
    record_stride=11
)
# rsync -avP user@remote:/path/to/file /local/path


 20%|██████████████▎                                                         | 9907/50000 [00:40<02:45, 242.54it/s]


KeyboardInterrupt: 

In [ ]:
# Optional: validate neighbors for a random neuron
idx = 123
print("neighbors:", engine.weights.get_neighbors(idx))
